In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [22]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [2]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [3]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [4]:
import json

user_prompt = json.dumps(documents[0])

messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [6]:
from dotenv import load_dotenv
from openai import OpenAI
from evaluation_utils import llm_structured

load_dotenv()
openai_client = OpenAI()

In [7]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

In [8]:
result.questions

['What is Retrieval-Augmented Generation, and how does it help a model answer questions better?',
 'Why do people use RAG instead of just asking the language model directly?',
 'What limits do large language models have in this lesson, like with new facts or private documents?',
 'What are you building in this module, and what kind of data will it answer questions from?',
 'What will the first part of the module cover before you make the system more agent-like?']

In [9]:
usage.input_tokens

1020

# Q1: Average Number of Input Tokens across 3 first documents

In [10]:
for i in range(3):
    print(documents[i]['filename'])

01-agentic-rag/lessons/01-intro.md
01-agentic-rag/lessons/02-environment.md
01-agentic-rag/lessons/03-rag.md


In [12]:
# Loop over the documents
results = []
usages = []
for i in range(3):
    user_prompt = json.dumps(documents[i])

    result, usage = llm_structured(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results.append({
        "questions": result.questions,
        "filename": documents[i]['filename']
    })

    usages.append(usage)

In [16]:
n_tokens = 0

for usage in usages:
    n_tokens += usage.input_tokens

In [17]:
# Average
print(n_tokens / 3)

1353.0


# Q2: Full Ground Truth

In [2]:
import pandas as pd

df_ground_truth = pd.read_csv('ground-truth.csv')
ground_truth = df_ground_truth.to_dict(orient="records")

In [3]:
# Chunk Documents
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [4]:
len(chunks)

295

In [11]:
# Text Search
from minsearch import Index, VectorSearch
from embedder import Embedder

embed = Embedder()

text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

text_index.fit(chunks)

vector_index = VectorSearch(
    keyword_fields=['filename']
)
chunk_contents = [chunk['content'] for chunk in chunks]

X = embed.encode_batch(chunk_contents)
vector_index.fit(X, chunks)

def text_search(query: str, num_results: int):
    return text_index.search(
        query,
        num_results = num_results
    )

def vector_search(query: str, num_results: int):
    query_vector = embed.encode(query)
    return vector_index.search(
        query_vector,
        num_results = num_results
    )

In [8]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [9]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [14]:
q = ground_truth[0]["question"]
print(q)
text_search(q, 10)

What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?


[{'start': 3000,
  'content': 'we drop it.\n\nBuild a prompt that includes both the question and the context:\n\n```python\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\nInstead of sending the raw question to the LLM, we send this prompt:\n\n```python\nanswer = llm(prompt)\nprint(answer)\n```\n\nAfter that, the answer is correct: "Yes, you can still join. If you want to\nreceive a certificate, you need to submit your project while\nsubmissions are still open."\n\nThis is the answer we actually want to give to our students. What we\njust did is nothing but RAG.\n\n## Retrieval plus generation\n\nRAG stands for Retrieval-Augmented Generation. Generation is the LLM\nproducing text, and retrieval is search. We retriev

# Q3: First Result with Vector Search

In [13]:
vector_search(q, 10)

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

# Q4: Evaluating Text Search

In [45]:
from tqdm.auto import tqdm

def compute_relevance(q, search_function, num_results):
    filename = q["filename"]
    results = search_function(query=q["question"], num_results=num_results)

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == filename))

    return relevance

def compute_relevance_total(ground_truth, search_function, num_results):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function, num_results)
        relevance_total.append(relevance)

    return relevance_total

In [46]:
relevance = compute_relevance_total(ground_truth, text_search, 10)

  0%|          | 0/360 [00:00<?, ?it/s]

In [24]:
# Hit Rate Function
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [25]:
hit_rate(relevance)

0.8416666666666667

# Q5: Vector Search Evaluations

In [26]:
vector_relevance = compute_relevance_total(ground_truth, vector_search, 10)

  0%|          | 0/360 [00:00<?, ?it/s]

In [27]:
hit_rate(vector_relevance)

0.8361111111111111

In [28]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [29]:
print(f"Vector Search - MRR: {mrr(vector_relevance)}")

Vector Search - MRR: 0.5646472663139328


# Q6: Tuning Hybrid Search

In [40]:
# Hybrid Search

from tqdm.auto import tqdm

def compute_relevance_hybrid(q, search_function, k):
    filename = q["filename"]
    results = search_function(query=q["question"], k=k)

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == filename))

    return relevance

def compute_relevance_total_hybrid(ground_truth, search_function, k=k):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_hybrid(q, search_function, k=k)
        relevance_total.append(relevance)

    return relevance_total

In [41]:
k = [1, 50, 100, 200]
mrr_list = []

for val in k:
    relevance = compute_relevance_total_hybrid(ground_truth, hybrid_search, k=val)
    mrr_value = mrr(relevance)
    mrr_list.append({
        "k": val,
        "mrr": mrr_value
    })

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

In [44]:
import json

print(json.dumps(mrr_list))

[{"k": 1, "mrr": 0.6481944444444449}, {"k": 50, "mrr": 0.637916666666667}, {"k": 100, "mrr": 0.637916666666667}, {"k": 200, "mrr": 0.637916666666667}]


Max MRR generated when `k=1`